# Public API consumer contracts

## What this shows
A consumer-facing smoke playground for the package root and selected subpackages. It demonstrates the public import style that downstream notebooks and applications should use: `import siege_utilities as su` plus selected public subpackages.

## Why it matters
The regression authority remains in hard tests and scripts, not this notebook. This notebook makes the intended call shape readable and executable offline so documentation cannot drift into private imports or fail-open examples.

## Prereqs
Pure Python only. No credentials, network, GDAL, GeoPandas, or live geocoder calls.

## Next
Use `tests/test_public_api_surface.py`, `scripts/check_lazy_imports.py`, and the API contract checks when changing public symbols.


## 1. Import through public surfaces

Consumers should import the package root and public subpackages. Private implementation modules are not needed for ordinary use.


In [ ]:
import siege_utilities as su
import siege_utilities.geo as geo

print('siege_utilities version:', su.__version__)
print('root has GeocodingError:', hasattr(su, 'GeocodingError'))
print('geo has GeocodingError :', hasattr(geo, 'GeocodingError'))


## 2. Geocoding core contracts are stdlib-safe

These helpers do not require pandas, geopy, network access, GDAL, or credentials. They are safe for pure/offline notebooks and for applications that want to validate inputs before choosing a live provider.


In [ ]:
assert su.GeocodingError is geo.GeocodingError
assert issubclass(su.GeocodingError, RuntimeError)

country_examples = {
    'US': su.get_country_name('US'),
    'United States': su.get_country_code('United States'),
    'countries_sample': list(su.list_countries())[:5],
}
address = su.concatenate_addresses('301 W 2nd St', 'Austin', 'TX', '78701')

print(country_examples)
print(address)


## 3. Dependency-gated live geocoding is explicit

`get_coordinates` and `use_nominatim_geocoder` are live/provider helpers. They require optional dependencies and, for real calls, network/provider availability. This playground does **not** call Nominatim. Instead it shows the introspection contract and the typed error boundary so unavailable optional surfaces are not mistaken for usable functions.


In [ ]:
info = su.get_package_info()
optional = info['optional_dependency_symbols']

for name in ['get_coordinates', 'use_nominatim_geocoder']:
    meta = optional.get(name, {})
    print(name, {
        'available': meta.get('available', name in info['available_functions']),
        'required_dependencies': meta.get('required_dependencies', []),
        'missing_dependencies': meta.get('missing_dependencies', []),
        'category': meta.get('category'),
    })

try:
    raise su.GeocodingError('provider returned no match for the fixture address')
except su.GeocodingError as exc:
    print('typed geocoding boundary:', exc)


## 4. Regression authority

This notebook is a consumer-contract playground. Hard regression authority lives in:

- `tests/test_public_api_surface.py` — public `__all__`, promoted symbol resolution, and geocoding degraded-mode contracts.
- `scripts/check_lazy_imports.py` — lazy registry structural integrity with explicit optional-dependency skip counts.
- API contract checks in CI, including the public API regression job and lazy import integrity job.

Do not use silent fail-open examples. Provider no-match, provider failure, and missing dependency states should be visible and typed, not swallowed into an empty frame.


In [ ]:
required = ['GeocodingError', 'get_country_name', 'get_country_code', 'list_countries', 'concatenate_addresses']
missing = [name for name in required if not hasattr(su, name)]
assert not missing, missing
assert 'GeocodingError' in su.__all__
print('public API playground checks passed')


## Related

- Issue: `#1225` public API consumer-contract playground.
- Parent audit: `#1176` public API review.
- Geocoding substrate: `#1220` and `#1228`.
- Notebook execution authority: `tests/test_notebooks.py` pure notebook tier.
